In [ ]:
!pip install openai

In [ ]:
API_KEY="use your own key"

In [ ]:
from openai import OpenAI
from typing import List, Dict
from collections import Counter
import textwrap

In [ ]:
def call_solar_api(
    prompt: str,
    api_key: str = API_KEY,
    model: str = "solar-mini-250422",
    temperature: float = 0.0,
):
    client = OpenAI(
        api_key=api_key,
        base_url="https://api.upstage.ai/v1"
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        stream=False,
        temperature=temperature
    )

    return response.choices[0].message.content

In [ ]:
question="안녕, 네 소개를 해줘"

In [ ]:
result = call_solar_api(prompt=question)
print(result)

안녕하세요! 저는 Upstage의 AI 챗봇인 Solar입니다. 저는 여러분이 궁금한 것들을 해결하고, 다양한 주제에 대해 대화할 수 있도록 도와드리기 위해 여기에 있습니다. 저는 스마트하고 유쾌하며 재미있는 답변을 제공하기 위해 노력합니다.


In [ ]:
question_sample = [
    {
        "question": "글루코스는 근육 세포로 어떻게 운반되는가?",
        "choices": [
            "A. GLUT4라고 불리는 단백질 운반체를 통해.",
            "B. 인슐린의 존재 하에만.",
            "C. 헥소키나아제를 통해.",
            "D. 단일 탄수화물 산 운반체를 통해."
        ],
        "answer": "A"
    },
    {
        "question": "What is the capital of France?",
        "choices": [
            "A. Berlin",
            "B. Paris",
            "C. Rome",
            "D. Madrid"
        ],
        "answer": "B"
    },
    {
        "question": "2 + 2 = ?",
        "choices": [
            "A. 3",
            "B. 4",
            "C. 5",
            "D. 6"
        ],
        "answer": "B"
    }
]

In [ ]:
# 문제 평가 함수

def evaluate_single_mmlu_question(
    question_data: Dict,
    api_key: str,
    model: str = "solar-mini-250422",
) -> bool:
    choices = question_data["choices"]
    choices_text = "\n".join(choices)
    prompt = question_data["question"] + "\n" + choices_text # 현재는 문제와 옵션을 단순히 합해서 대입
    print("prompt:")
    print(prompt)
    response = call_solar_api(
        prompt=prompt,
        api_key=api_key,
        model=model
    )

    # 예측값 파싱 (첫 글자만 사용)
    print("----모델 답변----")
    print(response)
    print("---------------")
    pred = response.strip().upper()[0]
    gt = question_data["answer"]

    return pred == gt

In [ ]:
def evaluate_mmlu_dataset(
    dataset: List[Dict],
    api_key: str,
    model: str = "solar-mini-250422",
):
    correct = 0
    total = len(dataset)

    for i, item in enumerate(dataset):
        is_correct = evaluate_single_mmlu_question(
            item, api_key, model
        )
        if is_correct:
            correct += 1

        print(f"[{i+1}/{total}] {'✓' if is_correct else '✗'}")

    accuracy = correct / total
    return accuracy

In [ ]:
accuracy = evaluate_mmlu_dataset(
    dataset=question_sample,
    api_key=API_KEY
)

print(f"\nAccuracy: {accuracy:.2%}")

prompt:
글루코스는 근육 세포로 어떻게 운반되는가?
A. GLUT4라고 불리는 단백질 운반체를 통해.
B. 인슐린의 존재 하에만.
C. 헥소키나아제를 통해.
D. 단일 탄수화물 산 운반체를 통해.
----모델 답변----
정답은 A. GLUT4라고 불리는 단백질 운반체를 통해입니다.

글루코스가 근육 세포로 운반되는 과정을 자세히 설명하면 다음과 같습니다:

1. **인슐린 신호전달**: 혈당 수치가 상승하면, 췌장은 인슐린을 분비합니다. 인슐린은 근육 세포의 표면에 있는 인슐린 수용체에 결합합니다.

2. **GLUT4 운반체의 이동**: 인슐린이 수용체에 결합하면, 세포 내에서 신호전달 경로가 활성화되어 GLUT4 운반체가 세포막으로 이동하게 됩니다. GLUT4는 글루코스를 세포 내로 운반하는 주요 운반체입니다.

3. **글루코스 운반**: GLUT4 운반체가 세포막에 위치하면, 글루코스가 혈류에서 세포 내로 확산됩니다. GLUT4는 글루코스에 대한 높은 친화성을 가지고 있어 효율적인 운반을 가능하게 합니다.

4. **글리코겐 합성 및 에너지 생산**: 세포 내로 들어온 글루코스는 에너지원으로 사용되거나, 미래의 사용을 위해 글리코겐으로 저장됩니다.

B 옵션은 잘못된 것입니다. 인슐린은 GLUT4 운반체의 이동을 촉진하지만, 글루코스 운반은 인슐린의 존재 하에만 일어나는 것은 아닙니다. 운동과 같은 다른 자극도 GLUT4 운반체를 세포막으로 이동시킬 수 있습니다.

C 옵션은 잘못된 것입니다. 헥소키나아제는 글루코스를 포도당-6-인산으로 전환하는 효소로, 세포 내로 들어온 글루코스의 대사에 관여하지만 운반에는 관여하지 않습니다.

D 옵션은 잘못된 것입니다. 단일 탄수화물 산 운반체(MCTs)는 주로 단당류와 같은 작은 유기산을 운반하는 데 관여하며, 글루코스 운반에는 관여하지 않습니다.

따라서, 글루코스가 근육 세포로 운반되는 주요 메커니즘은 GLUT4 운반체를 통한 것입니다.
---------------
[1/3] ✗
prompt:
What

## MMLU 정답 맞추기

In [ ]:
# formatting 함수 설정 / "respond with only the letter"을 추가하여 하나의 알파벳으로 답하도록 지정
def format_mmlu_prompt(question: str, choices: List[str]) -> str:
    choices_text = "\n".join(choices)

    return f"""
Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
{question}
Choices:
{choices_text}
"""

In [ ]:
# 문제 평가 함수

def evaluate_single_mmlu_question(
    question_data: Dict,
    api_key: str,
    model: str = "solar-mini-250422",
) -> bool:

    prompt = format_mmlu_prompt(question_data["question"], question_data["choices"]) # formatting prompt 추가
    print("prompt:")
    print(prompt)
    response = call_solar_api(
        prompt=prompt,
        api_key=api_key,
        model=model
    )

    # 예측값 파싱 (첫 글자만 사용)
    print("----모델 답변----")
    print(response)
    print("---------------")

    pred = response.strip().upper()[0]
    gt = question_data["answer"]

    return pred == gt

In [ ]:
accuracy = evaluate_mmlu_dataset(
    dataset=question_sample,
    api_key=API_KEY
)

print(f"\nMMLU Accuracy: {accuracy:.2%}")

prompt:

Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
글루코스는 근육 세포로 어떻게 운반되는가?
Choices:
A. GLUT4라고 불리는 단백질 운반체를 통해.
B. 인슐린의 존재 하에만.
C. 헥소키나아제를 통해.
D. 단일 탄수화물 산 운반체를 통해.

----모델 답변----
A. GLUT4라고 불리는 단백질 운반체를 통해.
---------------
[1/3] ✓
prompt:

Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
What is the capital of France?
Choices:
A. Berlin
B. Paris
C. Rome
D. Madrid

----모델 답변----
B.
---------------
[2/3] ✓
prompt:

Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
2 + 2 = ?
Choices:
A. 3
B. 4
C. 5
D. 6

----모델 답변----
B. 4
---------------
[3/3] ✓

MMLU Accuracy: 100.00%


## Temperature 조절

In [ ]:
PROMPT_FREE = "Explain what prompt engineering is in 1 short sentence"

temps = [0.0, 0.3, 0.7, 1.0]
runs_per_temp = 3  # 각 temperature에서 몇 번 돌릴지

for t in temps:
    print("="*80)
    print(f"temperature = {t}")
    print("-"*80)
    for r in range(runs_per_temp):
        out = call_solar_api(PROMPT_FREE, API_KEY, temperature=t)
        print(f"\n[run {r+1}]")
        print(out)

temperature = 0.0
--------------------------------------------------------------------------------

[run 1]
Prompt engineering is the art and science of crafting precise and effective inputs to generate desired outputs from AI models, optimizing for accuracy, efficiency, and desired behavior.

[run 2]
Prompt engineering is the art and science of crafting precise and effective inputs to generate desired outputs from AI models, optimizing for accuracy, efficiency, and desired behavior.

[run 3]
Prompt engineering is the art and science of crafting precise and effective inputs to generate desired outputs from AI models, optimizing for accuracy, efficiency, and desired behavior.
temperature = 0.3
--------------------------------------------------------------------------------

[run 1]
Prompt engineering is the art and science of crafting precise and effective inputs to generate desired outputs from AI models, optimizing for accuracy, efficiency, and desired outcomes.

[run 2]
Prompt engine

In [ ]:
question = "글루코스는 근육 세포로 어떻게 운반되는가?"
choices = [
        "A. GLUT4라고 불리는 단백질 운반체를 통해.",
        "B. 인슐린의 존재 하에만.",
        "C. 헥소키나아제를 통해.",
        "D. 단일 탄수화물 산 운반체를 통해."
        ]
gold = "A"
prompt_mcq = format_mmlu_prompt(question, choices)

temps = [0.0, 0.5, 1.0, 2.0]
runs_per_temp = 30

for t in temps:
    answers = []
    for _ in range(runs_per_temp):
        out = call_solar_api(prompt_mcq, API_KEY, temperature=t)
        pred = out.strip().upper()[0]
        gt = gold
        answers.append(pred)
    dist = Counter(answers)
    correct = dist[gold]
    print(f"temperature={t:<3}  dist={dict(dist)}  accuracy={correct/runs_per_temp:.2%}")


temperature=0.0  dist={'A': 30}  accuracy=100.00%
temperature=0.5  dist={'A': 30}  accuracy=100.00%
temperature=1.0  dist={'A': 30}  accuracy=100.00%
temperature=2.0  dist={'A': 24, '해': 1, '글': 1, 'G': 2, 'T': 2}  accuracy=80.00%


## In-Context Learning

### Zero-Shot vs Few-Shot

In [ ]:
examples = [
    {
        "question": "다음 중 뉴턴의 운동 제2법칙을 나타내는 공식은?",
        "choices": [
            "A. F = m / a",
            "B. F = m * a",
            "C. E = mc^2",
            "D. v = d / t"
        ],
        "answer": "B"
    },
    {
        "question": "Python에서 리스트의 길이를 반환하는 내장 함수는 무엇인가?",
        "choices": [
            "A. size()",
            "B. count()",
            "C. length()",
            "D. len()"
        ],
        "answer": "D"
    }
]


In [ ]:
# 예시가 있을 경우 위쪽에 넣어주면서 답변의 형태 등을 보여줌

def format_mmlu_prompt(
    question: str,
    choices: List[str],
    examples: List[Dict] = None
) -> str:
    """
    question: 질문 텍스트
    choices: 보기 리스트
    examples: Few-shot 예시 리스트 (없으면 Zero-shot)
    """

    prompt_text = ""

    # Few-shot 예시가 있는 경우
    if examples:
        prompt_text += "Here are some examples of how to answer:\n\n"
        for ex in examples:
            ex_choices = "\n".join(ex["choices"])
            prompt_text += (
                f"Question:\n{ex['question']}\n"
                f"Choices:\n{ex_choices}\n"
                f"Answer:\n{ex['answer']}\n\n"
            )
        prompt_text += "---\nNow, answer the following question.\n\n"

    # 메인 질문
    choices_text = "\n".join(choices)
    prompt_text += (
        "Answer the following multiple-choice question.\n"
        "Respond with only the letter (A, B, C, or D).\n\n"
        f"Question:\n{question}\n"
        f"Choices:\n{choices_text}\n"
        "Answer:\n"
    )

    return prompt_text.strip()


### Zero-shot vs Few-shot Prompting

- **Zero-shot**
  - 예시 없이 바로 질문
  - 모델의 기본 능력에 의존

- **Few-shot**
  - 예시를 통해 답변 형식과 패턴을 유도


In [ ]:
# 타겟 문제 (Python 리스트 길이)
target_data = question_sample[0]

print("="*20 + " CASE 1: Zero-shot " + "="*20)

prompt_zero_shot = format_mmlu_prompt(
    target_data["question"],
    target_data["choices"]
)

print("[입력 프롬프트]")
print(prompt_zero_shot)


==================== CASE 1: Zero-shot ====================
[입력 프롬프트]
Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
글루코스는 근육 세포로 어떻게 운반되는가?
Choices:
A. GLUT4라고 불리는 단백질 운반체를 통해.
B. 인슐린의 존재 하에만.
C. 헥소키나아제를 통해.
D. 단일 탄수화물 산 운반체를 통해.
Answer:


In [ ]:
# Few-shot 예시 2개 사용
print("="*20 + " CASE 2: Few-shot " + "="*20)

prompt_few_shot = format_mmlu_prompt(
    target_data["question"],
    target_data["choices"],
    examples=examples
)

print("[입력 프롬프트]")
print(prompt_few_shot)


==================== CASE 2: Few-shot ====================
[입력 프롬프트]
Here are some examples of how to answer:

Question:
다음 중 뉴턴의 운동 제2법칙을 나타내는 공식은?
Choices:
A. F = m / a
B. F = m * a
C. E = mc^2
D. v = d / t
Answer:
B

Question:
Python에서 리스트의 길이를 반환하는 내장 함수는 무엇인가?
Choices:
A. size()
B. count()
C. length()
D. len()
Answer:
D

---
Now, answer the following question.

Answer the following multiple-choice question.
Respond with only the letter (A, B, C, or D).

Question:
글루코스는 근육 세포로 어떻게 운반되는가?
Choices:
A. GLUT4라고 불리는 단백질 운반체를 통해.
B. 인슐린의 존재 하에만.
C. 헥소키나아제를 통해.
D. 단일 탄수화물 산 운반체를 통해.
Answer:


In [ ]:
pred_zero = call_solar_api(prompt_zero_shot, API_KEY)
print(pred_zero)

A. GLUT4라고 불리는 단백질 운반체를 통해.


In [ ]:
pred_few = call_solar_api(prompt_few_shot, API_KEY)
print(pred_few)

A.
